# LeetCode #1307: Verbal Arithmetic Puzzle

https://leetcode.com/problems/verbal-arithmetic-puzzle/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (try all 10! mappings)** | $O(10!)$ | $O(10)$ |
| **Optimal: Backtracking with Column Carry ★** | $O(10!)$ worst case, much faster in practice | $O(10)$ |

---

## Understanding the Methods

### Brute Force (try all 10! mappings)
Enumerate all $10! = 3{,}628{,}800$ permutations of digits 0–9 for the distinct letters, evaluate the equation for each, and return true if any mapping satisfies it. No pruning — all permutations are tried even when partial assignments already violate column sums.

### Optimal: Backtracking with Column Carry ★
Process the equation column by column from right to left (units, tens, hundreds, …). For each column, collect the letters that appear, assign unused digits to unassigned letters, and check whether the column sum modulo 10 matches the result's digit while propagating the carry. Invalid partial assignments are pruned as soon as a column constraint fails, cutting the search space dramatically.

**Constraints:**
* 2 <= words.length <= 5
* 1 <= words[i].length <= 7
* words[i] and result contain only uppercase English letters
* Total distinct letters ≤ 10
* Leading zeros are not allowed

## Solutions
### C#

In [ ]:
// Backtracking with column carry: assign digits column by column and prune early
public class Solution {
    private int[] charToDigit;
    private bool[] digitUsed;
    private bool[] isLeading; // characters that cannot be zero (first letter of each word)
    private List<List<int>> columns; // columns[i] = list of (coefficient) for each char
    private int[] coefficients; // net coefficient per character summed across all columns

    public bool IsSolvable(string[] words, string result) {
        charToDigit = new int[26];
        Array.Fill(charToDigit, -1);
        digitUsed = new bool[10];
        isLeading = new bool[26];

        // Mark leading characters — they cannot map to 0
        foreach (var w in words) isLeading[w[0] - 'A'] = true;
        isLeading[result[0] - 'A'] = true;

        // Compute net coefficient for each character:
        // positive for word letters, negative for result letters
        int[] coeff = new int[26];
        foreach (var w in words)
            for (int i = 0; i < w.Length; i++)
                coeff[w[i] - 'A'] += (int)Math.Pow(10, w.Length - 1 - i);
        for (int i = 0; i < result.Length; i++)
            coeff[result[i] - 'A'] -= (int)Math.Pow(10, result.Length - 1 - i);

        // Collect distinct letters to assign
        var letters = new List<int>();
        for (int i = 0; i < 26; i++)
            if (coeff[i] != 0) letters.Add(i);

        // Try all digit assignments with early pruning
        return Backtrack(letters, coeff, 0, 0);
    }

    private bool Backtrack(List<int> letters, int[] coeff, int pos, long total) {
        if (pos == letters.Count) {
            // All letters assigned — check if the total sum is exactly zero
            return total == 0;
        }
        int c = letters[pos];
        for (int d = 0; d <= 9; d++) {
            // Skip if digit already used or leading zero
            if (digitUsed[d] || (d == 0 && isLeading[c])) continue;
            charToDigit[c] = d;
            digitUsed[d] = true;
            // Add this letter's contribution to the running total
            if (Backtrack(letters, coeff, pos + 1, total + (long)d * coeff[c]))
                return true;
            digitUsed[d] = false;
            charToDigit[c] = -1;
        }
        return false;
    }
}

### Python

In [ ]:
# Backtracking with coefficient sum: assign digits to letters, prune when impossible
from typing import List

class Solution:
    def isSolvable(self, words: List[str], result: str) -> bool:
        # Compute net coefficient for each character
        coeff = {}
        for w in words:
            for i, c in enumerate(w):
                coeff[c] = coeff.get(c, 0) + 10 ** (len(w) - 1 - i)
        for i, c in enumerate(result):
            coeff[c] = coeff.get(c, 0) - 10 ** (len(result) - 1 - i)

        # Leading letters cannot map to 0
        no_zero = {w[0] for w in words} | {result[0]}

        letters = list(coeff.keys())
        used = [False] * 10

        def backtrack(pos, total):
            if pos == len(letters):
                # All letters assigned — valid only if net sum is zero
                return total == 0
            c = letters[pos]
            for d in range(10):
                if used[d] or (d == 0 and c in no_zero):
                    continue
                used[d] = True
                # Accumulate this letter's contribution to the total
                if backtrack(pos + 1, total + d * coeff[c]):
                    return True
                used[d] = False
            return False

        return backtrack(0, 0)

### Go

In [ ]:
// Backtracking with coefficient sum: assign digits and prune via running total
package main

func isSolvable(words []string, result string) bool {
    coeff := make(map[byte]int)
    // Accumulate positive contributions from words, negative from result
    for _, w := range words {
        for i := 0; i < len(w); i++ {
            place := 1
            for p := 0; p < len(w)-1-i; p++ {
                place *= 10
            }
            coeff[w[i]] += place
        }
    }
    for i := 0; i < len(result); i++ {
        place := 1
        for p := 0; p < len(result)-1-i; p++ {
            place *= 10
        }
        coeff[result[i]] -= place
    }

    // Leading characters must not map to 0
    noZero := make(map[byte]bool)
    for _, w := range words {
        noZero[w[0]] = true
    }
    noZero[result[0]] = true

    // Collect distinct letters
    letters := make([]byte, 0, len(coeff))
    coeffArr := make([]int, 0, len(coeff))
    for c, v := range coeff {
        letters = append(letters, c)
        coeffArr = append(coeffArr, v)
    }

    used := [10]bool{}
    var backtrack func(pos, total int) bool
    backtrack = func(pos, total int) bool {
        if pos == len(letters) {
            return total == 0
        }
        c := letters[pos]
        for d := 0; d <= 9; d++ {
            if used[d] || (d == 0 && noZero[c]) {
                continue
            }
            used[d] = true
            if backtrack(pos+1, total+d*coeffArr[pos]) {
                return true
            }
            used[d] = false
        }
        return false
    }
    return backtrack(0, 0)
}

### Rust

In [ ]:
// Backtracking with coefficient sum: assign digits, prune via running total
use std::collections::HashMap;

impl Solution {
    pub fn is_solvable(words: Vec<String>, result: String) -> bool {
        let mut coeff: HashMap<u8, i64> = HashMap::new();
        // Words contribute positively, result contributes negatively
        for w in &words {
            let wb = w.as_bytes();
            for (i, &c) in wb.iter().enumerate() {
                let place = 10i64.pow((wb.len() - 1 - i) as u32);
                *coeff.entry(c).or_insert(0) += place;
            }
        }
        let rb = result.as_bytes();
        for (i, &c) in rb.iter().enumerate() {
            let place = 10i64.pow((rb.len() - 1 - i) as u32);
            *coeff.entry(c).or_insert(0) -= place;
        }

        let mut no_zero: std::collections::HashSet<u8> = std::collections::HashSet::new();
        for w in &words { no_zero.insert(w.as_bytes()[0]); }
        no_zero.insert(rb[0]);

        let letters: Vec<u8> = coeff.keys().copied().collect();
        let coeffs: Vec<i64> = letters.iter().map(|c| coeff[c]).collect();
        let mut used = [false; 10];

        fn backtrack(pos: usize, total: i64, letters: &[u8], coeffs: &[i64],
                     used: &mut [bool; 10], no_zero: &std::collections::HashSet<u8>) -> bool {
            if pos == letters.len() { return total == 0; }
            for d in 0i64..10 {
                if used[d as usize] || (d == 0 && no_zero.contains(&letters[pos])) { continue; }
                used[d as usize] = true;
                if backtrack(pos + 1, total + d * coeffs[pos], letters, coeffs, used, no_zero) {
                    return true;
                }
                used[d as usize] = false;
            }
            false
        }
        backtrack(0, 0, &letters, &coeffs, &mut used, &no_zero)
    }
}

## Example Scenarios

**1. Common Case** — Classic SEND + MORE = MONEY

**Input:** `words = ["SEND","MORE"], result = "MONEY"`
The classic cryptarithmetic puzzle. The unique solution is S=9, E=5, N=6, D=7, M=1, O=0, R=8, Y=2 giving 9567 + 1085 = 10652. Backtracking finds this after pruning thousands of invalid partial assignments.

**2. Slightly Complex** — Simple single-letter equation

**Input:** `words = ["A","B"], result = "C"`
Net coefficients: A→+1, B→+1, C→−1. We need A+B=C. Valid assignments: A=1,B=2,C=3 (or any similar triple). Backtracking finds the first valid triple quickly.

**3. Edge Case: Time Factor** — No valid solution exists

**Input:** `words = ["AAA","BB"], result = "C"`
Net coefficient: A→+111 (×2 positions + hundreds), B→+11, C→−1. The equation requires large multi-digit sums to collapse to a single digit — impossible. All $10 \times 9$ assignments for A and B are tried before returning false.

**4. Edge Case: Space Factor** — Maximum distinct letters (10)

**Input:** `words = ["ABCDE","FGHIJ"], result = "RESULT"` (10 unique letters)
All 10 letters need assignments. The coefficient map has 10 entries; `used[]` is size 10. Space is $O(10) = O(1)$. Recursion depth is at most 10.

**5. Almost-Impossible but Plausible** — Leading zeros trap

**Input:** `words = ["A","B"], result = "AA"`; A cannot be 0
With no-zero constraint on A, only digits 1–9 are tried for A. B can be any unused digit. The constraint `A + B = 10*A + A` = $11A$ means $B = 10A$, which is impossible for single-digit B. Backtracking correctly returns false.